# NYC GREEN TAXI 2025 PROJECT

## verifing catalogs and creating schemas

In [0]:
spark.sql("SHOW CATALOGS").show(truncate=False)

+---------+
|catalog  |
+---------+
|dbacademy|
|samples  |
|system   |
|workspace|
+---------+



In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.taxi_project;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.taxi_project.bronze_trips (
    trip_id BIGINT,
    pickup_datetime TIMESTAMP,
    dropoff_datetime TIMESTAMP,
    passenger_count INT,
    trip_distance DOUBLE,
    fare_amount DECIMAL(10, 2),
    total_amount DECIMAL(10, 2)
)
USING DELTA;

In [0]:
%sql
SHOW VOLUMES IN workspace.taxi_project;

database,volume_name
taxi_project,green_taxi_raw
taxi_project,locations


In [0]:
%sql
DESCRIBE volume workspace.taxi_project.green_taxi_raw;

name,catalog,database,owner,storage_location,volume_type,comment,securable_type,securable_kind
green_taxi_raw,workspace,taxi_project,mohdmohiuddin1409@gmail.com,,MANAGED,null,VOLUME,VOLUME_DB_STORAGE


## Step1: Creating Volume

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS
workspace.taxi_project.green_taxi_raw;

In [0]:
%sql
show catalogs;

catalog
dbacademy
samples
system
workspace


## uploading data to volume


In [0]:
import urllib.request as req
import os

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-01.parquet"

volume_path = "/Volumes/workspace/taxi_project/green_taxi_raw"

destination = f"{volume_path}/green_tripdata_2025-01.parquet"

req.urlretrieve(url,destination)

print("download complete")

print(f"file size:",os.path.getsize(destination), "bytes")



download complete
file size: 1178451 bytes


## validating files in volume

In [0]:
file_path = "/Volumes/workspace/taxi_project/green_taxi_raw/green_tripdata_2025-01.parquet"

df = spark.read.parquet(file_path)

df.show(5)
df.printSchema()

print("Total rows:", df.count())

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|cbd_congestion_fee|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+
|       2| 2025-01-01 00:03:01|  2025-01-01 00:17:12|                 N|         1|          75|         2

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/taxi_project/green_taxi_raw"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/taxi_project/green_taxi_raw/green_tripdata_2025-01.parquet,green_tripdata_2025-01.parquet,1178451,1792854851000
dbfs:/Volumes/workspace/taxi_project/green_taxi_raw/green_tripdata_2025-02.parquet,green_tripdata_2025-02.parquet,1117474,1792854872000
dbfs:/Volumes/workspace/taxi_project/green_taxi_raw/green_tripdata_2025-03.parquet,green_tripdata_2025-03.parquet,1253510,1792854872000
dbfs:/Volumes/workspace/taxi_project/green_taxi_raw/green_tripdata_2025-04.parquet,green_tripdata_2025-04.parquet,1269227,1792854873000
dbfs:/Volumes/workspace/taxi_project/green_taxi_raw/green_tripdata_2025-05.parquet,green_tripdata_2025-05.parquet,1338798,1792854873000
dbfs:/Volumes/workspace/taxi_project/green_taxi_raw/green_tripdata_2025-06.parquet,green_tripdata_2025-06.parquet,1212934,1792854873000
dbfs:/Volumes/workspace/taxi_project/green_taxi_raw/green_tripdata_2025-07.parquet,green_tripdata_2025-07.parquet,1188380,1792854873000
dbfs:/Volumes/workspace/taxi_project/green_taxi_raw/green_tripdata_2025-08.parquet,green_tripdata_2025-08.parquet,1149707,1792854873000
dbfs:/Volumes/workspace/taxi_project/green_taxi_raw/green_tripdata_2025-09.parquet,green_tripdata_2025-09.parquet,1209831,1792854873000
dbfs:/Volumes/workspace/taxi_project/green_taxi_raw/green_tripdata_2025-10.parquet,green_tripdata_2025-10.parquet,1205685,1792854874000


## Uploading all 12 files to volume

In [0]:
import urllib.request as r
from pathlib import Path

volume_path = "/Volumes/workspace/taxi_project/green_taxi_raw"
base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/"

for month in range(2, 13):  # Feb to dec 2025
    filename = f"green_tripdata_2025-{month:02d}.parquet"
    url = base_url + filename
    destination = f"{volume_path}/{filename}"

    if Path(destination).exists():
        print(f"Already exists, skipping: {filename}")
        continue

    try:
        r.urlretrieve(url, destination)
        print(f"Download successful: {filename}")

    except Exception as e:
        print(f"Download failed: {filename}")
        print(f"Reason: {e}")        



Already exists, skipping: green_tripdata_2025-02.parquet
Already exists, skipping: green_tripdata_2025-03.parquet
Already exists, skipping: green_tripdata_2025-04.parquet
Already exists, skipping: green_tripdata_2025-05.parquet
Already exists, skipping: green_tripdata_2025-06.parquet
Already exists, skipping: green_tripdata_2025-07.parquet
Already exists, skipping: green_tripdata_2025-08.parquet
Already exists, skipping: green_tripdata_2025-09.parquet
Already exists, skipping: green_tripdata_2025-10.parquet
Already exists, skipping: green_tripdata_2025-11.parquet
Already exists, skipping: green_tripdata_2025-12.parquet


In [0]:
df = spark.read.parquet("/Volumes/workspace/taxi_project/green_taxi_raw/*.parquet")
df.show(5)

print(f"rows: {df.count()}")
print(f"columns: {len(df.columns)}")
df.printSchema()

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|cbd_congestion_fee|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+
|       2| 2025-05-01 00:17:04|  2025-05-01 00:56:06|                 N|         1|          25|         2

## Writing data to bronze conatiner 


In [0]:
bronze_table = "workspace.taxi_project.bronze_trips"

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(bronze_table)

In [0]:
%sql
DROP TABLE IF EXISTS workspace.taxi_project.bronze_trips;

In [0]:
%sql
select * from workspace.taxi_project.bronze_trips

VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
2,2025-01-01T00:03:01.000,2025-01-01T00:17:12.000,N,1,75,235,1,5.93,24.7,1.0,0.5,6.8,0.0,null,1.0,34.0,1,1,0.0,0.0
2,2025-01-01T00:19:59.000,2025-01-01T00:25:52.000,N,1,166,75,1,1.32,8.6,1.0,0.5,0.0,0.0,null,1.0,11.1,2,1,0.0,0.0
2,2025-01-01T00:05:29.000,2025-01-01T00:07:21.000,N,5,171,73,1,0.41,25.55,0.0,0.0,0.0,0.0,null,1.0,26.55,2,2,0.0,0.0
2,2025-01-01T00:52:24.000,2025-01-01T01:07:52.000,N,1,74,223,1,4.12,21.2,1.0,0.5,6.13,6.94,null,1.0,36.77,1,1,0.0,0.0
2,2025-01-01T00:25:05.000,2025-01-01T01:01:10.000,N,1,66,158,1,4.71,33.8,1.0,0.5,7.81,0.0,null,1.0,46.86,1,1,2.75,0.0
2,2025-01-01T01:03:45.000,2025-01-01T01:42:09.000,N,1,260,260,1,4.26,31.0,1.0,0.5,0.0,0.0,null,1.0,33.5,2,1,0.0,0.0
2,2025-01-01T00:28:12.000,2025-01-01T00:37:53.000,N,1,7,202,1,2.0,11.4,1.0,0.5,22.0,0.0,null,1.0,35.9,1,1,0.0,0.0
2,2025-01-01T00:25:38.000,2025-01-01T00:36:47.000,N,1,166,244,1,2.09,13.5,1.0,0.5,0.0,0.0,null,1.0,16.0,2,1,0.0,0.0
2,2025-01-01T00:52:04.000,2025-01-01T00:58:36.000,N,1,166,151,1,0.95,7.9,1.0,0.5,2.08,0.0,null,1.0,12.48,1,1,0.0,0.0
2,2024-12-31T22:42:13.000,2024-12-31T22:42:31.000,N,3,74,74,1,0.06,23.0,1.0,0.0,0.0,0.0,null,1.0,25.0,2,1,0.0,0.0


In [0]:
bronze_df = spark.read.table("workspace.taxi_project.bronze_trips")


## data quality audit


### null check

In [0]:
from pyspark.sql.functions import *

nulls = bronze_df.select([count(when(col(c).isNull(), 1)).alias(c) for c in bronze_df.columns])
display(nulls)


VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,0,0,49880,49880,0,0,49880,0,0,0,0,0,0,591375,0,0,49880,49927,49880,3824


In [0]:
from pyspark.sql.functions import *

null_columns = []

for c in bronze_df.columns:
    result = count(when(col(c).isNull(), 1)).alias(c)
    null_columns.append(result)
display(bronze_df.select(null_columns))

count(CASE WHEN (VendorID IS NULL) THEN 1 END),count(CASE WHEN (passenger_count IS NULL) THEN 1 END)
0,49880


### Quality Summary

In [0]:
from pyspark.sql.functions import *

quality_summary = bronze_df.select(

    count("*").alias("row_count"),

    sum(when(
        col("lpep_pickup_datetime").isNull(),1)\
            .otherwise(0))\
                .alias("nulls_pickup_datetime"),

    sum(when(
        col("lpep_dropoff_datetime").isNull(),1)\
            .otherwise(0))\
                .alias("nulls_dropoff_datetime"), 

    count(when(
        col("trip_distance")<=0,1))\
            .alias("non_positive_distance"), 

    sum(when(col("total_amount").isNull(),1).otherwise(0)).alias("nulls_total_amount"))


display(quality_summary)

row_count,nulls_pickup_datetime,nulls_dropoff_datetime,non_positive_distance,nulls_total_amount
591375,0,0,24438,0


### Checking all numericals negetive values

In [0]:
from pyspark.sql.functions import *

null_columns = []

i=['VendorID',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'ehail_fee',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'trip_type',
 'congestion_surcharge',
 'cbd_congestion_fee']

for c in i:
    result = sum(when(col(c)<0, 1).otherwise(0)).alias(c)
    null_columns.append(result)
display(bronze_df.select(null_columns))

VendorID,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,0,0,0,0,0,1736,688,1534,68,3,0,1770,1774,0,0,52,15


## Checking Duplicate Records

In [0]:
print("Total rows:", bronze_df.count())
print("Distinct complete rows:", bronze_df.distinct().count())

Total rows: 591375
Distinct complete rows: 591375


### What are we checking?
- dropoff_before_pickup: A trip appears to finish before it starts.
- zero_duration_trips: Pickup and drop-off timestamps are identical. These need investigation, not automatic deletion.
- negative_distance: Distance is negative, which is suspicious.
- negative_passenger_count: Passenger count is negative, which is suspicious.


In [0]:
from pyspark.sql.functions import *

quality_checks = bronze_df.select(
    count("*").alias("total_rows"),

    sum(when(
        col("lpep_dropoff_datetime") <
        col("lpep_pickup_datetime"), 1
    ).otherwise(0)).alias("dropoff_before_pickup"),

    sum(when(
        col("lpep_dropoff_datetime") ==
        col("lpep_pickup_datetime"), 1
    ).otherwise(0)).alias("zero_duration_trips"),

    sum(when(
        col("trip_distance") < 0, 1
    ).otherwise(0)).alias("negative_distance"),

    sum(when(
        col("passenger_count") < 0, 1
    ).otherwise(0)).alias("negative_passenger_count")
)

display(quality_checks)

total_rows,dropoff_before_pickup,zero_duration_trips,negative_distance,negative_passenger_count
591375,1344,566,0,0


### Step 2: Create the Silver DataFrame
Once the checks are done, we'll create a cleaned DataFrame from Bronze. For now, let's apply only conservative rules:
1. Keep records with valid pickup and drop-off timestamps.
2. Keep records where drop-off is not earlier than pickup.
3. Remove negative trip distances.
4. Preserve zero-distance trips for now.
5. Preserve negative fares and adjustments until we understand why they're present.

In [0]:
silver_df = (
    bronze_df
    .filter(
        col("lpep_pickup_datetime").isNotNull()
        & col("lpep_dropoff_datetime").isNotNull()
        & (
            col("lpep_dropoff_datetime")
            >= col("lpep_pickup_datetime")
        )
        & (
            col("trip_distance").isNull()
            | (col("trip_distance") >= 0)
        )
    )
)
display(silver_df.limit(15))

VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
2,2025-01-01T00:03:01.000,2025-01-01T00:17:12.000,N,1,75,235,1,5.93,24.7,1.0,0.5,6.8,0.0,null,1.0,34.0,1,1,0.0,0.0
2,2025-01-01T00:19:59.000,2025-01-01T00:25:52.000,N,1,166,75,1,1.32,8.6,1.0,0.5,0.0,0.0,null,1.0,11.1,2,1,0.0,0.0
2,2025-01-01T00:05:29.000,2025-01-01T00:07:21.000,N,5,171,73,1,0.41,25.55,0.0,0.0,0.0,0.0,null,1.0,26.55,2,2,0.0,0.0
2,2025-01-01T00:52:24.000,2025-01-01T01:07:52.000,N,1,74,223,1,4.12,21.2,1.0,0.5,6.13,6.94,null,1.0,36.77,1,1,0.0,0.0
2,2025-01-01T00:25:05.000,2025-01-01T01:01:10.000,N,1,66,158,1,4.71,33.8,1.0,0.5,7.81,0.0,null,1.0,46.86,1,1,2.75,0.0
2,2025-01-01T01:03:45.000,2025-01-01T01:42:09.000,N,1,260,260,1,4.26,31.0,1.0,0.5,0.0,0.0,null,1.0,33.5,2,1,0.0,0.0
2,2025-01-01T00:28:12.000,2025-01-01T00:37:53.000,N,1,7,202,1,2.0,11.4,1.0,0.5,22.0,0.0,null,1.0,35.9,1,1,0.0,0.0
2,2025-01-01T00:25:38.000,2025-01-01T00:36:47.000,N,1,166,244,1,2.09,13.5,1.0,0.5,0.0,0.0,null,1.0,16.0,2,1,0.0,0.0
2,2025-01-01T00:52:04.000,2025-01-01T00:58:36.000,N,1,166,151,1,0.95,7.9,1.0,0.5,2.08,0.0,null,1.0,12.48,1,1,0.0,0.0
2,2024-12-31T22:42:13.000,2024-12-31T22:42:31.000,N,3,74,74,1,0.06,23.0,1.0,0.0,0.0,0.0,null,1.0,25.0,2,1,0.0,0.0


In [0]:
print(f"bronze_row_count: {bronze_df.count()}")
print(f"silver_row_count: {silver_df.count()}")
print(f"comparison of rows: {bronze_df.count() - silver_df.count()}")

bronze_row_count: 591375
silver_row_count: 590031
comparison of rows: 1344


removed 1344 rows

In [0]:
removed_rows = bronze_df.filter(col("lpep_dropoff_datetime") < col("lpep_pickup_datetime"))
removed_rows.count()
removed_rows.show(20)

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|cbd_congestion_fee|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+
|       1| 2025-02-01 16:30:00|  2025-02-01 16:22:34|                 N|        99|         145|         1

In [0]:
from pyspark.sql.functions import *

bronze_df.filter(
    col("lpep_dropoff_datetime") <
    col("lpep_pickup_datetime")
).count()

1344

# Saving Silver Table


In [0]:
silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.taxi_project.silver_trips")

In [0]:
silver_df = spark.table("workspace.taxi_project.silver_trips")

print("Silver rows:", silver_df.count())
print("Columns:", len(silver_df.columns))

display(silver_df.limit(10))

Silver rows: 590031
Columns: 21


VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
2,2025-01-01T00:03:01.000,2025-01-01T00:17:12.000,N,1,75,235,1,5.93,24.7,1.0,0.5,6.8,0.0,null,1.0,34.0,1,1,0.0,0.0
2,2025-01-01T00:19:59.000,2025-01-01T00:25:52.000,N,1,166,75,1,1.32,8.6,1.0,0.5,0.0,0.0,null,1.0,11.1,2,1,0.0,0.0
2,2025-01-01T00:05:29.000,2025-01-01T00:07:21.000,N,5,171,73,1,0.41,25.55,0.0,0.0,0.0,0.0,null,1.0,26.55,2,2,0.0,0.0
2,2025-01-01T00:52:24.000,2025-01-01T01:07:52.000,N,1,74,223,1,4.12,21.2,1.0,0.5,6.13,6.94,null,1.0,36.77,1,1,0.0,0.0
2,2025-01-01T00:25:05.000,2025-01-01T01:01:10.000,N,1,66,158,1,4.71,33.8,1.0,0.5,7.81,0.0,null,1.0,46.86,1,1,2.75,0.0
2,2025-01-01T01:03:45.000,2025-01-01T01:42:09.000,N,1,260,260,1,4.26,31.0,1.0,0.5,0.0,0.0,null,1.0,33.5,2,1,0.0,0.0
2,2025-01-01T00:28:12.000,2025-01-01T00:37:53.000,N,1,7,202,1,2.0,11.4,1.0,0.5,22.0,0.0,null,1.0,35.9,1,1,0.0,0.0
2,2025-01-01T00:25:38.000,2025-01-01T00:36:47.000,N,1,166,244,1,2.09,13.5,1.0,0.5,0.0,0.0,null,1.0,16.0,2,1,0.0,0.0
2,2025-01-01T00:52:04.000,2025-01-01T00:58:36.000,N,1,166,151,1,0.95,7.9,1.0,0.5,2.08,0.0,null,1.0,12.48,1,1,0.0,0.0
2,2024-12-31T22:42:13.000,2024-12-31T22:42:31.000,N,3,74,74,1,0.06,23.0,1.0,0.0,0.0,0.0,null,1.0,25.0,2,1,0.0,0.0


### Next, we'll use PySpark to create useful features such as:
1. trip_duration_minutes: Calculate how long each trip lasted.
2. pickup_hour: Extract the pickup hour for peak-hour analysis.
3. pickup_day: Identify weekdays and weekends.
4. pickup_month: Prepare monthly trip analysis.

In [0]:
silver_df = spark.read.table("workspace.taxi_project.silver_trips")

In [0]:
silver_df.show(3)

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|cbd_congestion_fee|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+
|       2| 2025-01-01 00:03:01|  2025-01-01 00:17:12|                 N|         1|          75|         2

In [0]:
from pyspark.sql.functions import *

silver_df = silver_df.withColumn(
            "trip_duration_minutes", 
                    (unix_timestamp(col("lpep_dropoff_datetime")) -
                    unix_timestamp(col("lpep_pickup_datetime")))/60
        ).withColumn("pickup_hour",
                    hour("lpep_pickup_datetime")
        ).withColumn("pickup_day",
                    date_format("lpep_pickup_datetime","EEEE")
        ).withColumn("pickup_month",
                    month("lpep_pickup_datetime"))


display(silver_df.limit(10))


VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,trip_duration_minutes,pickup_hour,pickup_day,pickup_month
2,2025-01-01T00:03:01.000,2025-01-01T00:17:12.000,N,1,75,235,1,5.93,24.7,1.0,0.5,6.8,0.0,null,1.0,34.0,1,1,0.0,0.0,14.183333333333334,0,Wednesday,1
2,2025-01-01T00:19:59.000,2025-01-01T00:25:52.000,N,1,166,75,1,1.32,8.6,1.0,0.5,0.0,0.0,null,1.0,11.1,2,1,0.0,0.0,5.883333333333334,0,Wednesday,1
2,2025-01-01T00:05:29.000,2025-01-01T00:07:21.000,N,5,171,73,1,0.41,25.55,0.0,0.0,0.0,0.0,null,1.0,26.55,2,2,0.0,0.0,1.8666666666666667,0,Wednesday,1
2,2025-01-01T00:52:24.000,2025-01-01T01:07:52.000,N,1,74,223,1,4.12,21.2,1.0,0.5,6.13,6.94,null,1.0,36.77,1,1,0.0,0.0,15.466666666666667,0,Wednesday,1
2,2025-01-01T00:25:05.000,2025-01-01T01:01:10.000,N,1,66,158,1,4.71,33.8,1.0,0.5,7.81,0.0,null,1.0,46.86,1,1,2.75,0.0,36.083333333333336,0,Wednesday,1
2,2025-01-01T01:03:45.000,2025-01-01T01:42:09.000,N,1,260,260,1,4.26,31.0,1.0,0.5,0.0,0.0,null,1.0,33.5,2,1,0.0,0.0,38.4,1,Wednesday,1
2,2025-01-01T00:28:12.000,2025-01-01T00:37:53.000,N,1,7,202,1,2.0,11.4,1.0,0.5,22.0,0.0,null,1.0,35.9,1,1,0.0,0.0,9.683333333333334,0,Wednesday,1
2,2025-01-01T00:25:38.000,2025-01-01T00:36:47.000,N,1,166,244,1,2.09,13.5,1.0,0.5,0.0,0.0,null,1.0,16.0,2,1,0.0,0.0,11.15,0,Wednesday,1
2,2025-01-01T00:52:04.000,2025-01-01T00:58:36.000,N,1,166,151,1,0.95,7.9,1.0,0.5,2.08,0.0,null,1.0,12.48,1,1,0.0,0.0,6.533333333333333,0,Wednesday,1
2,2024-12-31T22:42:13.000,2024-12-31T22:42:31.000,N,3,74,74,1,0.06,23.0,1.0,0.0,0.0,0.0,null,1.0,25.0,2,1,0.0,0.0,0.3,22,Tuesday,12


### Now lets validate the new columns
we are doing all this checks on "trip_duration_mins".
- Negative duration: a possible data-quality problem.
- Zero duration: could be a genuine edge case or an issue worth investigating.
- Over 180 minutes: potentially unusual, but not automatically invalid.
- NULL duration: the duration couldn't be calculated.

In [0]:
silver_df.select(
    count("*").alias("all_trips"),
    sum(when(col("trip_duration_minutes")<0,1).otherwise(0)).alias("negetive_duration"),
    sum(when(col("trip_duration_minutes")==0,1).otherwise(0)).alias("Zero_durations"),
    sum(when(col("trip_duration_minutes").isNull(),1).otherwise(0)).alias("null_duration")
).display()

all_trips,negetive_duration,Zero_durations,null_duration
590031,0,566,0


Negative duration = 0: Great. Your Silver filter removed trips where drop-off occurred before pickup.

Zero duration = 566: These trips have identical pickup and drop-off timestamps. They may be unusual records, but we shouldn't delete them automatically.

NULL duration = 0: Every trip has a calculated duration.

In [0]:
display(
    silver_df
    .filter(col("trip_duration_minutes") == 0)
    .select(
        "lpep_pickup_datetime",
        "lpep_dropoff_datetime",
        "trip_distance",
        "passenger_count",
        "fare_amount",
        "total_amount"
    )
    .limit(20)
)

lpep_pickup_datetime,lpep_dropoff_datetime,trip_distance,passenger_count,fare_amount,total_amount
2025-01-01T05:14:17.000,2025-01-01T05:14:17.000,0.0,1,3.0,5.5
2025-01-01T05:14:21.000,2025-01-01T05:14:21.000,0.0,1,3.0,12.5
2025-01-02T08:55:46.000,2025-01-02T08:55:46.000,0.0,0,65.75,65.75
2025-01-03T00:36:28.000,2025-01-03T00:36:28.000,0.0,1,3.0,43.5
2025-01-03T10:29:10.000,2025-01-03T10:29:10.000,0.0,1,17.7,21.95
2025-01-03T22:50:02.000,2025-01-03T22:50:02.000,0.0,1,3.0,40.5
2025-01-05T20:58:33.000,2025-01-05T20:58:33.000,0.0,1,19.5,21.0
2025-01-06T01:48:32.000,2025-01-06T01:48:32.000,0.0,1,3.0,5.5
2025-01-06T13:52:35.000,2025-01-06T13:52:35.000,0.0,1,70.0,71.5
2025-01-06T21:14:37.000,2025-01-06T21:14:37.000,0.0,1,9.3,11.8


### Check all 566 records before deciding
Let's calculate how many zero-duration trips have zero distance versus positive distance, and examine passenger counts.

In [0]:
from pyspark.sql.functions import *

zero_duration_df = silver_df.filter(
    col("trip_duration_minutes") == 0
)

zero_duration_df.select(
    count("*").alias("total_zero_duration"),

    sum(when(
        col("trip_distance") == 0, 1
    ).otherwise(0)).alias("zero_distance"),

    sum(when(
        col("trip_distance") > 0, 1
    ).otherwise(0)).alias("positive_distance"),

    sum(when(
        col("passenger_count") == 0, 1
    ).otherwise(0)).alias("zero_passenger_count")
).show()

+-------------------+-------------+-----------------+--------------------+
|total_zero_duration|zero_distance|positive_distance|zero_passenger_count|
+-------------------+-------------+-----------------+--------------------+
|                566|          466|              100|                  52|
+-------------------+-------------+-----------------+--------------------+



### 1. What do these results tell us?
- 466 trips: Both duration and distance are zero. These records deserve investigation, but we can't prove they're invalid from those fields alone.
- 100 trips: Duration is zero, but distance is positive. This is a stronger signal of inconsistent or insufficient timestamp data.
- 52 trips: Passenger count is zero. This can mean missing or unrecorded passenger information, not necessarily that the trip itself is invalid.
### 2. Should we remove the 566 records?
Not yet. For your project, let's preserve them in Silver and document the issue. We can also create a quality flag so that downstream analysis can exclude them when appropriate.

### Adding a Zero duration flag

In [0]:
from pyspark.sql.functions import *

silver_df = silver_df.withColumn(
    "zero_duration_flag",
    when(col("trip_duration_minutes") == 0, True)
    .otherwise(False)
)

In [0]:
display(silver_df.limit(10))

VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,trip_duration_minutes,pickup_hour,pickup_day,pickup_month,zero_duration_flag
2,2025-01-01T00:03:01.000,2025-01-01T00:17:12.000,N,1,75,235,1,5.93,24.7,1.0,0.5,6.8,0.0,null,1.0,34.0,1,1,0.0,0.0,14.183333333333334,0,Wednesday,1,false
2,2025-01-01T00:19:59.000,2025-01-01T00:25:52.000,N,1,166,75,1,1.32,8.6,1.0,0.5,0.0,0.0,null,1.0,11.1,2,1,0.0,0.0,5.883333333333334,0,Wednesday,1,false
2,2025-01-01T00:05:29.000,2025-01-01T00:07:21.000,N,5,171,73,1,0.41,25.55,0.0,0.0,0.0,0.0,null,1.0,26.55,2,2,0.0,0.0,1.8666666666666667,0,Wednesday,1,false
2,2025-01-01T00:52:24.000,2025-01-01T01:07:52.000,N,1,74,223,1,4.12,21.2,1.0,0.5,6.13,6.94,null,1.0,36.77,1,1,0.0,0.0,15.466666666666667,0,Wednesday,1,false
2,2025-01-01T00:25:05.000,2025-01-01T01:01:10.000,N,1,66,158,1,4.71,33.8,1.0,0.5,7.81,0.0,null,1.0,46.86,1,1,2.75,0.0,36.083333333333336,0,Wednesday,1,false
2,2025-01-01T01:03:45.000,2025-01-01T01:42:09.000,N,1,260,260,1,4.26,31.0,1.0,0.5,0.0,0.0,null,1.0,33.5,2,1,0.0,0.0,38.4,1,Wednesday,1,false
2,2025-01-01T00:28:12.000,2025-01-01T00:37:53.000,N,1,7,202,1,2.0,11.4,1.0,0.5,22.0,0.0,null,1.0,35.9,1,1,0.0,0.0,9.683333333333334,0,Wednesday,1,false
2,2025-01-01T00:25:38.000,2025-01-01T00:36:47.000,N,1,166,244,1,2.09,13.5,1.0,0.5,0.0,0.0,null,1.0,16.0,2,1,0.0,0.0,11.15,0,Wednesday,1,false
2,2025-01-01T00:52:04.000,2025-01-01T00:58:36.000,N,1,166,151,1,0.95,7.9,1.0,0.5,2.08,0.0,null,1.0,12.48,1,1,0.0,0.0,6.533333333333333,0,Wednesday,1,false
2,2024-12-31T22:42:13.000,2024-12-31T22:42:31.000,N,3,74,74,1,0.06,23.0,1.0,0.0,0.0,0.0,null,1.0,25.0,2,1,0.0,0.0,0.3,22,Tuesday,12,false


## saving to silver_trips

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.taxi_project.silver_trips")

# GOLD TABLE

### Next, let's build our first Gold table: hourly trip analysis. We'll calculate:
- Total trips by pickup hour
- Average trip duration by pickup hour
- Average trip distance by pickup hour
- Total passenger charges by pickup hour

In [0]:
from pyspark.sql.functions import *

silver_df = spark.read.table("workspace.taxi_project.silver_trips")

hourly_gold_df = silver_df.groupBy("pickup_hour").agg(count("*").alias("total_trips"),round(avg("trip_duration_minutes"),2).alias("avg_duration_minutes"),round(avg("trip_distance"),2).alias("avg_trip_distance"),round(sum("total_amount"),2).alias("total_passenger_charges")).orderBy("pickup_hour")
display(hourly_gold_df)

pickup_hour,total_trips,avg_duration_minutes,avg_trip_distance,total_passenger_charges
0,9712,19.59,3.37,275053.04
1,6483,18.88,3.2,182993.71
2,4597,20.68,3.48,148378.03
3,3626,20.78,3.51,124576.1
4,3559,21.37,13.17,136766.92
5,4180,17.76,46.56,137704.25
6,10698,21.81,15.5,270123.49
7,22871,25.84,52.77,547003.33
8,29734,22.06,39.71,726084.27
9,31630,23.51,33.26,775271.09


### saving hourly gold table

In [0]:
hourly_gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.taxi_project.gold_hourly_trips")

In [0]:
gold_df = spark.table("workspace.taxi_project.gold_hourly_trips")

gold_df.printSchema()
display(gold_df)

root
 |-- pickup_hour: integer (nullable = true)
 |-- total_trips: long (nullable = true)
 |-- avg_duration_minutes: double (nullable = true)
 |-- avg_trip_distance: double (nullable = true)
 |-- total_passenger_charges: double (nullable = true)



pickup_hour,total_trips,avg_duration_minutes,avg_trip_distance,total_passenger_charges
0,9712,19.59,3.37,275053.04
1,6484,18.88,3.2,183066.65
2,4597,20.68,3.48,148378.03
3,3626,20.78,3.51,124576.1
4,3559,21.37,13.17,136766.92
5,4180,17.76,46.56,137704.25
6,10698,21.81,15.5,270123.49
7,22871,25.84,52.77,547003.33
8,29734,22.06,39.71,726084.27
9,31630,23.51,33.26,775271.09


### one thing stands out in your results: average trip distance reaches 52.77 at 7 AM, and 46.56 at 5 AM. Those values are unusually high for average distances in a taxi dataset, so let's inspect them before drawing business conclusions.

In [0]:
from pyspark.sql.functions import *

silver_df = spark.table("workspace.taxi_project.silver_trips")

silver_df.select(
    min("trip_distance").alias("min_distance"),
    expr("percentile_approx(trip_distance, 0.50)")
        .alias("median_distance"),
    expr("percentile_approx(trip_distance, 0.95)")
        .alias("p95_distance"),
    max("trip_distance").alias("max_distance")
).show()

+------------+---------------+------------+------------+
|min_distance|median_distance|p95_distance|max_distance|
+------------+---------------+------------+------------+
|         0.0|           1.98|        9.79|   262315.94|
+------------+---------------+------------+------------+



In [0]:
from pyspark.sql.functions import *

silver_df.filter(
    col("trip_distance") > 100
).select(
    "VendorID",
    "lpep_pickup_datetime",
    "lpep_dropoff_datetime",
    "trip_distance",
    "trip_duration_minutes",
    "fare_amount",
    "total_amount",
    "PULocationID",
    "DOLocationID"
).orderBy(
    col("trip_distance")
).show(20, truncate=False)

+--------+--------------------+---------------------+-------------+---------------------+-----------+------------+------------+------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|trip_distance|trip_duration_minutes|fare_amount|total_amount|PULocationID|DOLocationID|
+--------+--------------------+---------------------+-------------+---------------------+-----------+------------+------------+------------+
|2       |2025-04-20 21:05:25 |2025-04-20 22:56:21  |101.53       |110.93333333333334   |320.0      |381.0       |130         |265         |
|2       |2025-06-01 02:27:00 |2025-06-01 05:27:49  |102.98       |180.81666666666666   |350.0      |386.0       |80          |265         |
|2       |2025-07-21 08:21:49 |2025-07-21 10:52:56  |110.72       |151.11666666666667   |745.0      |746.0       |74          |265         |
|2       |2025-08-28 15:59:27 |2025-08-28 19:04:36  |119.45       |185.15               |801.0      |802.5       |93          |265         |
|2       |202

In [0]:
from pyspark.sql.functions import *

silver_df = silver_df.withColumn(
    "distance_quality_flag",
    when(col("trip_distance") > 100, "SUSPICIOUS")
    .otherwise("VALID_OR_REVIEW")
)
silver_df.show(5)


+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+---------------------+-----------+----------+------------+------------------+---------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|cbd_congestion_fee|trip_duration_minutes|pickup_hour|pickup_day|pickup_month|zero_duration_flag|distance_quality_flag|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+-------

In [0]:
silver_df.write.format("delta")\
    .mode("overwrite").option("overwriteSchema",True).saveAsTable("workspace.taxi_project.silver_trips")

### Revising the Gold Table After Identifying Suspicious Trip Distances

During data quality analysis, we identified 208 records with trip distances greater than 100 miles. These records are considered suspicious and require further investigation, as they may affect the accuracy of our analytical results.

Since the Gold table contains aggregated metrics, including the average trip distance, these unusually high values can skew the calculated averages and lead to misleading insights.

Therefore, we will review the identified records, apply the appropriate data quality rules, and recalculate the Gold table aggregations to ensure that the average trip distance and other analytical metrics are more accurate and reliable.

In [0]:
spark.table("workspace.taxi_project.silver_trips") \
    .groupBy("distance_quality_flag") \
    .count() \
    .show()

+---------------------+------+
|distance_quality_flag| count|
+---------------------+------+
|      VALID_OR_REVIEW|589823|
|           SUSPICIOUS|   208|
+---------------------+------+



# Recreating Gold Hourly trips

In [0]:
from pyspark.sql.functions import *

silver_df = spark.table("workspace.taxi_project.silver_trips")

hourly_gold_df = (
    silver_df
    .filter(col("distance_quality_flag") != "SUSPICIOUS")
    .groupBy("pickup_hour")
    .agg(
        count("*").alias("total_trips"),
        round(avg("trip_duration_minutes"), 2)
            .alias("avg_duration_minutes"),
        round(avg("trip_distance"), 2)
            .alias("avg_trip_distance"),
        round(sum("total_amount"), 2)
            .alias("total_passenger_charges")
    )
    .orderBy("pickup_hour")
)

display(hourly_gold_df)

pickup_hour,total_trips,avg_duration_minutes,avg_trip_distance,total_passenger_charges
0,9711,19.57,3.36,273957.5
1,6483,18.88,3.2,182993.71
2,4596,20.64,3.46,147992.03
3,3626,20.78,3.51,124576.1
4,3558,21.37,4.15,136724.66
5,4176,17.73,4.61,137470.15
6,10693,21.82,3.87,269955.66
7,22854,25.84,3.29,545886.13
8,29710,22.05,3.12,724564.09
9,31612,23.51,3.14,774335.61


In [0]:
from pyspark.sql.functions import *

display(
    hourly_gold_df.orderBy(
        col("total_trips").desc()
    ).limit(1)
)

display(
    hourly_gold_df.orderBy(
        col("avg_trip_distance").desc()
    ).limit(1)
)

display(
    hourly_gold_df.orderBy(
        col("avg_duration_minutes").desc()
    ).limit(1)
)

pickup_hour,total_trips,avg_duration_minutes,avg_trip_distance,total_passenger_charges
17,47046,19.62,2.73,1165776.78


pickup_hour,total_trips,avg_duration_minutes,avg_trip_distance,total_passenger_charges
5,4176,17.73,4.61,137470.15


pickup_hour,total_trips,avg_duration_minutes,avg_trip_distance,total_passenger_charges
11,31186,26.24,3.23,783275.09


In [0]:
hourly_gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.taxi_project.gold_hourly_trips")

# 2. Payment Type Analysis Gold Table

In [0]:
from pyspark.sql.functions import *

silver_df = spark.table("workspace.taxi_project.silver_trips")

payment_gold_df = (
    silver_df
    .filter(col("distance_quality_flag") != "SUSPICIOUS")
    .groupBy("payment_type")
    .agg(
        count("*").alias("total_trips"),
        round(avg("total_amount"), 2).alias("avg_trip_amount"),
        round(sum("total_amount"), 2).alias("total_revenue"),
        round(avg("tip_amount"), 2).alias("avg_tip")
    )
    .orderBy(col("total_revenue").desc())
)

display(payment_gold_df)

payment_type,total_trips,avg_trip_amount,total_revenue,avg_tip
1,406591,25.92,1.054064195E7,3.69
2,129816,21.71,2818473.5,0.0
null,48344,30.74,1486263.46,1.54
3,3784,4.72,17878.94,0.0
4,1244,3.54,4401.23,0.0
5,23,13.58,312.41,0.0


### Key business insights:
- Credit cards dominate: approximately 69% of all trips in this filtered dataset use payment type 1.
- Credit cards generate the most revenue: around $10.54 million.
- Cash tips show $0 average: this doesn't necessarily mean cash passengers never tip. The dataset may not capture cash tips in the same way as credit-card tips.
- NULL payment type needs investigation: 48,344 records have no payment type, so don't silently classify them as cash or credit card.

In [0]:
payment_gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.taxi_project.gold_payment_analysis")

### now lets investigate the null payment type 

In [0]:
silver_df.filter(col("payment_type").isNull()).select("payment_type","total_amount","tip_amount","trip_distance").display()

payment_type,total_amount,tip_amount,trip_distance
null,26.91,6.15,2.51
null,19.82,2.0,2.69
null,27.46,4.58,2.59
null,76.42,12.74,8.2
null,17.23,1.57,1.35
null,29.77,3.88,3.55
null,48.25,8.04,7.41
null,26.62,4.44,2.83
null,34.31,5.72,4.75
null,40.64,6.77,5.41


# 3. Creating Gold pickuploc analysis Table

In [0]:
from pyspark.sql.functions import *

silver_df = spark.table("workspace.taxi_project.silver_trips")

pickup_gold = silver_df.filter(col("distance_quality_flag") != "SUSPICIOUS")\
    .groupBy("PULocationID")\
    .agg(
        round(avg("total_amount"),2).alias("avg_trip_amount"),
          round(sum("total_amount"),2).alias("total_revenue"),
           count("*").alias("total_trips"),
            round(avg("trip_distance"),2).alias("avg_trip_distance")
            )\
    .orderBy(col("total_trips").desc())

display(pickup_gold)

       

PULocationID,avg_trip_amount,total_revenue,total_trips,avg_trip_distance
74,21.56,3126846.4,145057,2.36
75,21.53,1798163.36,83516,2.32
43,23.37,646846.89,27673,2.3
95,22.74,607956.44,26736,2.91
166,22.73,601838.43,26477,2.61
41,19.66,464326.64,23613,2.33
82,24.62,534706.89,21720,3.05
97,23.38,467869.32,20008,2.52
130,31.2,570312.35,18278,4.74
65,26.89,439769.5,16354,3.13


In [0]:
pickup_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.taxi_project.gold_pickup_analysis")

# Now lets give name to location ID

create volume for csv lookup file for location names

In [0]:
%sql
create volume workspace.taxi_project.locations

In [0]:
loc_names = spark.read.format("csv").option("header","true").option("inferschema","true").load("/Volumes/workspace/taxi_project/locations/loc_names.csv")
loc_names.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [0]:
loc_names.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.taxi_project.loc_names_table")

In [0]:
%sql
select * from workspace.taxi_project.loc_names_table

LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


In [0]:
%sql
select * from workspace.taxi_project.gold_pickup_analysis

PULocationID,avg_trip_amount,total_revenue,total_trips,avg_trip_distance
74,21.56,3127056.63,145068,2.36
75,21.53,1798200.47,83519,2.32
43,23.37,646846.89,27673,2.3
95,22.74,607956.44,26736,2.91
166,22.73,601849.31,26478,2.61
41,19.66,464333.94,23614,2.33
82,24.62,534720.21,21721,3.05
97,23.38,467869.32,20008,2.52
130,31.2,570312.35,18278,4.74
65,26.89,439769.5,16354,3.13


In [0]:
%sql
drop table workspace.taxi_project.gold_pickup_analysis
    


In [0]:
from pyspark.sql.functions import *

gold_pickup_df = spark.table("workspace.taxi_project.gold_pickup_analysis")
loc_df = spark.table("workspace.taxi_project.loc_names_table")
loc_df = loc_df.withColumnRenamed("LocationID", "PULocationID")

loc_add = gold_pickup_df.join(loc_df, on = "PULocationID", how = "left")
display(loc_add.limit(5))

PULocationID,avg_trip_amount,total_revenue,total_trips,avg_trip_distance,Borough,Zone,service_zone
74,21.56,3126846.4,145057,2.36,Manhattan,East Harlem North,Boro Zone
75,21.53,1798163.36,83516,2.32,Manhattan,East Harlem South,Boro Zone
43,23.37,646846.89,27673,2.3,Manhattan,Central Park,Yellow Zone
95,22.74,607956.44,26736,2.91,Queens,Forest Hills,Boro Zone
166,22.73,601838.43,26477,2.61,Manhattan,Morningside Heights,Boro Zone


## Save the new loc gold pickup table instead of old gold_pickup_analysis

In [0]:
loc_add.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.taxi_project.gold_pickup_analysis")

# 4.Gold Table - Daily trip Trends

In [0]:
from pyspark.sql.functions import *

silver_df = spark.table("workspace.taxi_project.silver_trips")

daily_gold_df = (
    silver_df
    .filter(col("distance_quality_flag") != "SUSPICIOUS")
    .withColumn(
        "pickup_date",
        to_date(col("lpep_pickup_datetime"))
    )
    .groupBy("pickup_date")
    .agg(
        count("*").alias("total_trips"),
        round(avg("trip_distance"), 2)
            .alias("avg_trip_distance"),
        round(avg("total_amount"), 2)
            .alias("avg_trip_amount"),
        round(sum("total_amount"), 2)
            .alias("total_revenue")
    )
    .orderBy("pickup_date")
)

display(daily_gold_df)

pickup_date,total_trips,avg_trip_distance,avg_trip_amount,total_revenue
2025-01-01,1001,2.94,25.05,25076.51
2025-01-02,1514,2.58,22.59,34204.76
2025-01-03,1524,2.73,23.95,36497.02
2025-01-04,1249,2.71,22.31,27869.2
2025-01-05,1129,2.79,22.87,25823.69
2025-01-06,1541,2.38,22.1,34049.13
2025-01-07,1648,2.42,22.36,36844.24
2025-01-08,1676,2.43,22.55,37796.27
2025-01-09,1868,2.46,22.77,42532.87
2025-01-10,1631,2.52,22.83,37236.56


In [0]:
daily_gold_df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.taxi_project.gold_daily_trends")

### Data Quality Issue: Unexpected Pickup Years

While creating the Gold tables, we discovered a record with a pickup year of 2008. To investigate this issue, we re-examined the complete Bronze table and identified **21 records outside the target year of 2025**:

- **1 record** from 2008
- **6 records** from 2024
- **14 records** from 2026

Since this project focuses exclusively on NYC Green Taxi data for 2025, these 21 records fall outside the scope of our analysis.

**Corrective Action:**
1. Recreate the Silver table by filtering records to include only trips with a pickup date in 2025, while preserving the existing data-cleaning rules, derived columns, and quality flags.
2. Validate the corrected Silver table to confirm that it contains only 2025 pickup records.
3. Rebuild and update all previously created Gold tables using the corrected Silver data to ensure that the analytical results reflect only the target year.

The Bronze table will remain unchanged to preserve the original ingested data.

In [0]:
daily_gold_df.filter(
    year(col("pickup_date")) == 2008
).show(truncate=False)

+-----------+-----------+-----------------+---------------+-------------+
|pickup_date|total_trips|avg_trip_distance|avg_trip_amount|total_revenue|
+-----------+-----------+-----------------+---------------+-------------+
+-----------+-----------+-----------------+---------------+-------------+



In [0]:
silver_df.filter(
    year(col("lpep_pickup_datetime")) == 2008
).select(
    "lpep_pickup_datetime",
    "lpep_dropoff_datetime",
    "trip_distance",
    "total_amount"
).show(20, truncate=False)

+--------------------+---------------------+-------------+------------+
|lpep_pickup_datetime|lpep_dropoff_datetime|trip_distance|total_amount|
+--------------------+---------------------+-------------+------------+
+--------------------+---------------------+-------------+------------+



In [0]:
from pyspark.sql.functions import *

silver_df.filter(
    year(col("lpep_pickup_datetime")) != 2025
).groupBy(
    year(col("lpep_pickup_datetime")).alias("pickup_year")
).count().orderBy("pickup_year").show()

+-----------+-----+
|pickup_year|count|
+-----------+-----+
|       2008|    1|
|       2024|    6|
|       2026|   14|
+-----------+-----+



In [0]:
from pyspark.sql.functions import *

bronze_df = spark.table("workspace.taxi_project.bronze_trips")

bronze_df.filter(
    year(col("lpep_pickup_datetime")) != 2025
).groupBy(
    year(col("lpep_pickup_datetime")).alias("pickup_year")
).count().orderBy("pickup_year").show()

+-----------+-----+
|pickup_year|count|
+-----------+-----+
|       2008|    1|
|       2024|    6|
|       2026|   14|
+-----------+-----+



In [0]:
from pyspark.sql.functions import *

bronze_df = spark.table("workspace.taxi_project.bronze_trips")

bronze_df.filter(
    year(col("lpep_pickup_datetime")) != 2025
).select(
    "lpep_pickup_datetime",
    "lpep_dropoff_datetime",
    "trip_distance",
    "total_amount"
).orderBy("lpep_pickup_datetime").display()

lpep_pickup_datetime,lpep_dropoff_datetime,trip_distance,total_amount
2008-12-31T15:13:04.000,2008-12-31T17:25:57.000,10.17,103.9
2024-12-25T23:13:15.000,2024-12-25T23:13:17.000,0.0,43.2
2024-12-29T01:23:41.000,2024-12-29T01:41:08.000,10.15,72.94
2024-12-31T19:56:14.000,2024-12-31T20:03:33.000,1.0,13.32
2024-12-31T22:42:13.000,2024-12-31T22:42:31.000,0.06,25.0
2024-12-31T23:01:11.000,2024-12-31T23:04:29.000,0.66,13.81
2024-12-31T23:30:28.000,2024-12-31T23:35:39.000,1.04,11.9
2026-01-01T11:35:42.000,2026-01-01T11:47:45.000,4.3,21.3
2026-01-01T12:08:45.000,2026-01-01T12:11:48.000,0.7,7.3
2026-01-01T13:27:57.000,2026-01-01T13:36:01.000,1.97,13.6


# Correcting Silver Table

In [0]:
from pyspark.sql.functions import *

bronze_df = spark.table("workspace.taxi_project.bronze_trips")

silver_df = (
    bronze_df
    .filter(
        (year(col("lpep_pickup_datetime")) == 2025)
        & col("lpep_pickup_datetime").isNotNull()
        & col("lpep_dropoff_datetime").isNotNull()
        & (
            col("lpep_dropoff_datetime")
            >= col("lpep_pickup_datetime")
        )
        & (
            col("trip_distance").isNull()
            | (col("trip_distance") >= 0)
        )
    )
    .withColumn(
        "trip_duration_minutes",
        (
            unix_timestamp("lpep_dropoff_datetime")
            - unix_timestamp("lpep_pickup_datetime")
        ) / 60
    )
    .withColumn(
        "pickup_hour",
        hour("lpep_pickup_datetime")
    )
    .withColumn(
        "pickup_month",
        month("lpep_pickup_datetime")
    )
    .withColumn(
        "pickup_day",
        date_format("lpep_pickup_datetime", "EEEE")
    )
    .withColumn(
        "zero_duration_flag",
        when(col("trip_duration_minutes") == 0, True)
        .otherwise(False)
    )
    .withColumn(
        "distance_quality_flag",
        when(col("trip_distance") > 100, "SUSPICIOUS")
        .otherwise("VALID_OR_REVIEW")
    )
)

print("Filtered Silver row count:", silver_df.count())

Filtered Silver row count: 590010


In [0]:
silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.taxi_project.silver_trips")

# Lets rerun and update all Gold Tables

# Correction Done

### Analyzing daily trends

In [0]:
from pyspark.sql.functions import *

daily_gold_df = spark.table("workspace.taxi_project.gold_daily_trends")

print("Highest trip day:")
daily_gold_df.orderBy(
    col("total_trips").desc()
).show(1, truncate=False)

print("Highest revenue day:")
daily_gold_df.orderBy(
    col("total_revenue").desc()
).show(1, truncate=False)

Highest trip day:
+-----------+-----------+-----------------+---------------+-------------+
|pickup_date|total_trips|avg_trip_distance|avg_trip_amount|total_revenue|
+-----------+-----------+-----------------+---------------+-------------+
|2025-05-21 |2089       |2.8              |25.28          |52819.7      |
+-----------+-----------+-----------------+---------------+-------------+
only showing top 1 row
Highest revenue day:
+-----------+-----------+-----------------+---------------+-------------+
|pickup_date|total_trips|avg_trip_distance|avg_trip_amount|total_revenue|
+-----------+-----------+-----------------+---------------+-------------+
|2025-09-03 |1811       |4.02             |32.96          |59688.89     |
+-----------+-----------+-----------------+---------------+-------------+
only showing top 1 row


# Adding payment type and location names to silver table

In [0]:
silver_df = spark.table("workspace.taxi_project.silver_trips")
payment_type = spark.table("workspace.taxi_project.dim_payment_type")
loc_names = spark.table("workspace.taxi_project.loc_names_table")
loc_names = loc_names.withColumnRenamed("LocationID", "PULocationID")
loc_names.show(5)

display(silver_df.limit(10))

+------------+-------------+--------------------+------------+
|PULocationID|      Borough|                Zone|service_zone|
+------------+-------------+--------------------+------------+
|           1|          EWR|      Newark Airport|         EWR|
|           2|       Queens|         Jamaica Bay|   Boro Zone|
|           3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|           4|    Manhattan|       Alphabet City| Yellow Zone|
|           5|Staten Island|       Arden Heights|   Boro Zone|
+------------+-------------+--------------------+------------+
only showing top 5 rows


VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,trip_duration_minutes,pickup_hour,pickup_month,pickup_day,zero_duration_flag,distance_quality_flag
2,2025-01-01T00:03:01.000,2025-01-01T00:17:12.000,N,1,75,235,1,5.93,24.7,1.0,0.5,6.8,0.0,null,1.0,34.0,1,1,0.0,0.0,14.183333333333334,0,1,Wednesday,false,VALID_OR_REVIEW
2,2025-01-01T00:19:59.000,2025-01-01T00:25:52.000,N,1,166,75,1,1.32,8.6,1.0,0.5,0.0,0.0,null,1.0,11.1,2,1,0.0,0.0,5.883333333333334,0,1,Wednesday,false,VALID_OR_REVIEW
2,2025-01-01T00:05:29.000,2025-01-01T00:07:21.000,N,5,171,73,1,0.41,25.55,0.0,0.0,0.0,0.0,null,1.0,26.55,2,2,0.0,0.0,1.8666666666666667,0,1,Wednesday,false,VALID_OR_REVIEW
2,2025-01-01T00:52:24.000,2025-01-01T01:07:52.000,N,1,74,223,1,4.12,21.2,1.0,0.5,6.13,6.94,null,1.0,36.77,1,1,0.0,0.0,15.466666666666667,0,1,Wednesday,false,VALID_OR_REVIEW
2,2025-01-01T00:25:05.000,2025-01-01T01:01:10.000,N,1,66,158,1,4.71,33.8,1.0,0.5,7.81,0.0,null,1.0,46.86,1,1,2.75,0.0,36.083333333333336,0,1,Wednesday,false,VALID_OR_REVIEW
2,2025-01-01T01:03:45.000,2025-01-01T01:42:09.000,N,1,260,260,1,4.26,31.0,1.0,0.5,0.0,0.0,null,1.0,33.5,2,1,0.0,0.0,38.4,1,1,Wednesday,false,VALID_OR_REVIEW
2,2025-01-01T00:28:12.000,2025-01-01T00:37:53.000,N,1,7,202,1,2.0,11.4,1.0,0.5,22.0,0.0,null,1.0,35.9,1,1,0.0,0.0,9.683333333333334,0,1,Wednesday,false,VALID_OR_REVIEW
2,2025-01-01T00:25:38.000,2025-01-01T00:36:47.000,N,1,166,244,1,2.09,13.5,1.0,0.5,0.0,0.0,null,1.0,16.0,2,1,0.0,0.0,11.15,0,1,Wednesday,false,VALID_OR_REVIEW
2,2025-01-01T00:52:04.000,2025-01-01T00:58:36.000,N,1,166,151,1,0.95,7.9,1.0,0.5,2.08,0.0,null,1.0,12.48,1,1,0.0,0.0,6.533333333333333,0,1,Wednesday,false,VALID_OR_REVIEW
2,2025-01-01T00:18:31.000,2025-01-01T00:18:46.000,N,5,92,92,1,0.0,35.0,0.0,0.0,2.0,0.0,null,1.0,38.0,1,2,0.0,0.0,0.25,0,1,Wednesday,false,VALID_OR_REVIEW


In [0]:
join_silver_locs = silver_df.join(loc_names, on= "PULocationID", how = "left")
join_silver_paymenttype = join_silver_locs.join(payment_type, on = "payment_type", how = "left")
silver_df = join_silver_paymenttype


In [0]:
display(silver_df)

payment_type,PULocationID,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,trip_type,congestion_surcharge,cbd_congestion_fee,trip_duration_minutes,pickup_hour,pickup_month,pickup_day,zero_duration_flag,distance_quality_flag,Borough,Zone,service_zone,payment_type_name,Borough,Zone,service_zone,payment_type_name
1,75,2,2025-01-01T00:03:01.000,2025-01-01T00:17:12.000,N,1,235,1,5.93,24.7,1.0,0.5,6.8,0.0,null,1.0,34.0,1,0.0,0.0,14.183333333333334,0,1,Wednesday,false,VALID_OR_REVIEW,Manhattan,East Harlem South,Boro Zone,Credit Card,Manhattan,East Harlem South,Boro Zone,Credit Card
2,166,2,2025-01-01T00:19:59.000,2025-01-01T00:25:52.000,N,1,75,1,1.32,8.6,1.0,0.5,0.0,0.0,null,1.0,11.1,1,0.0,0.0,5.883333333333334,0,1,Wednesday,false,VALID_OR_REVIEW,Manhattan,Morningside Heights,Boro Zone,Cash,Manhattan,Morningside Heights,Boro Zone,Cash
2,171,2,2025-01-01T00:05:29.000,2025-01-01T00:07:21.000,N,5,73,1,0.41,25.55,0.0,0.0,0.0,0.0,null,1.0,26.55,2,0.0,0.0,1.8666666666666667,0,1,Wednesday,false,VALID_OR_REVIEW,Queens,Murray Hill-Queens,Boro Zone,Cash,Queens,Murray Hill-Queens,Boro Zone,Cash
1,74,2,2025-01-01T00:52:24.000,2025-01-01T01:07:52.000,N,1,223,1,4.12,21.2,1.0,0.5,6.13,6.94,null,1.0,36.77,1,0.0,0.0,15.466666666666667,0,1,Wednesday,false,VALID_OR_REVIEW,Manhattan,East Harlem North,Boro Zone,Credit Card,Manhattan,East Harlem North,Boro Zone,Credit Card
1,66,2,2025-01-01T00:25:05.000,2025-01-01T01:01:10.000,N,1,158,1,4.71,33.8,1.0,0.5,7.81,0.0,null,1.0,46.86,1,2.75,0.0,36.083333333333336,0,1,Wednesday,false,VALID_OR_REVIEW,Brooklyn,DUMBO/Vinegar Hill,Boro Zone,Credit Card,Brooklyn,DUMBO/Vinegar Hill,Boro Zone,Credit Card
2,260,2,2025-01-01T01:03:45.000,2025-01-01T01:42:09.000,N,1,260,1,4.26,31.0,1.0,0.5,0.0,0.0,null,1.0,33.5,1,0.0,0.0,38.4,1,1,Wednesday,false,VALID_OR_REVIEW,Queens,Woodside,Boro Zone,Cash,Queens,Woodside,Boro Zone,Cash
1,7,2,2025-01-01T00:28:12.000,2025-01-01T00:37:53.000,N,1,202,1,2.0,11.4,1.0,0.5,22.0,0.0,null,1.0,35.9,1,0.0,0.0,9.683333333333334,0,1,Wednesday,false,VALID_OR_REVIEW,Queens,Astoria,Boro Zone,Credit Card,Queens,Astoria,Boro Zone,Credit Card
2,166,2,2025-01-01T00:25:38.000,2025-01-01T00:36:47.000,N,1,244,1,2.09,13.5,1.0,0.5,0.0,0.0,null,1.0,16.0,1,0.0,0.0,11.15,0,1,Wednesday,false,VALID_OR_REVIEW,Manhattan,Morningside Heights,Boro Zone,Cash,Manhattan,Morningside Heights,Boro Zone,Cash
1,166,2,2025-01-01T00:52:04.000,2025-01-01T00:58:36.000,N,1,151,1,0.95,7.9,1.0,0.5,2.08,0.0,null,1.0,12.48,1,0.0,0.0,6.533333333333333,0,1,Wednesday,false,VALID_OR_REVIEW,Manhattan,Morningside Heights,Boro Zone,Credit Card,Manhattan,Morningside Heights,Boro Zone,Credit Card
1,92,2,2025-01-01T00:18:31.000,2025-01-01T00:18:46.000,N,5,92,1,0.0,35.0,0.0,0.0,2.0,0.0,null,1.0,38.0,2,0.0,0.0,0.25,0,1,Wednesday,false,VALID_OR_REVIEW,Queens,Flushing,Boro Zone,Credit Card,Queens,Flushing,Boro Zone,Credit Card


In [0]:
silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.taxi_project.silver_trips")

In [0]:
silver_df.count()

590010

In [0]:
for table in [
    "gold_hourly_trips",
    "gold_daily_trends",
    "gold_payment_analysis",
    "gold_pickup_analysis"
]:
    print(f"\n--- {table} ---")
    spark.table(f"workspace.taxi_project.{table}").printSchema()


--- gold_hourly_trips ---
root
 |-- pickup_hour: integer (nullable = true)
 |-- total_trips: long (nullable = true)
 |-- avg_duration_minutes: double (nullable = true)
 |-- avg_trip_distance: double (nullable = true)
 |-- total_passenger_charges: double (nullable = true)


--- gold_daily_trends ---
root
 |-- pickup_date: date (nullable = true)
 |-- total_trips: long (nullable = true)
 |-- avg_trip_distance: double (nullable = true)
 |-- avg_trip_amount: double (nullable = true)
 |-- total_revenue: double (nullable = true)


--- gold_payment_analysis ---
root
 |-- payment_type: long (nullable = true)
 |-- total_trips: long (nullable = true)
 |-- avg_trip_amount: double (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- avg_tip: double (nullable = true)
 |-- payment_type_name: string (nullable = true)


--- gold_pickup_analysis ---
root
 |-- PULocationID: integer (nullable = true)
 |-- avg_trip_amount: double (nullable = true)
 |-- total_revenue: double (nullable = true

# Preperation of Dim Tables for Dashboard

In [0]:
from pyspark.sql import functions as F

silver = spark.table("workspace.taxi_project.silver_trips")

# 1. Date dimension
dim_date = (
    silver
    .select(F.to_date("lpep_pickup_datetime").alias("pickup_date"))
    .filter(F.col("pickup_date").isNotNull())
    .distinct()
    .withColumn("year", F.year("pickup_date"))
    .withColumn("month_number", F.month("pickup_date"))
    .withColumn("month_name", F.date_format("pickup_date", "MMMM"))
)

dim_date.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.taxi_project.dim_date"
)

# 2. Payment type dimension
dim_payment = (
    silver.select("payment_type", "payment_type_name")
    .dropDuplicates()
)

dim_payment.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.taxi_project.dim_payment_type"
)

# 3. Vendor dimension
dim_vendor = (
    silver.select("VendorID")
    .distinct()
)

dim_vendor.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.taxi_project.dim_vendor"
)

# 4. Trip type dimension
dim_trip = (
    silver.select("trip_type")
    .distinct()
)

dim_trip.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.taxi_project.dim_trip_type"
)

# 5. Pickup location dimension
dim_location = (
    silver.select("PULocationID", "Zone", "Borough", "service_zone")
    .dropDuplicates(["PULocationID"])
)

dim_location.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.taxi_project.dim_location"
)

In [0]:
tables = [
    "dim_date",
    "dim_payment_type",
    "dim_vendor",
    "dim_trip_type",
    "dim_location"
]

for table in tables:
    print(f"\n========== {table} ==========")

    df = spark.table(f"workspace.taxi_project.{table}")

    display(df)


========== dim_date ==========


pickup_date,year,month_number,month_name
2025-06-25,2025,6,June
2025-05-04,2025,5,May
2025-12-25,2025,12,December
2025-09-23,2025,9,September
2025-01-14,2025,1,January
2025-05-31,2025,5,May
2025-03-08,2025,3,March
2025-06-03,2025,6,June
2025-12-23,2025,12,December
2025-02-23,2025,2,February



========== dim_payment_type ==========


payment_type,payment_type_name
5,Unknown
3,No Charge
2,Cash
null,null
1,Credit Card
4,Dispute



========== dim_vendor ==========


VendorID
2
6
1



========== dim_trip_type ==========


trip_type
1
2
null



========== dim_location ==========


PULocationID,Zone,Borough,service_zone
255,Williamsburg (North Side),Brooklyn,Boro Zone
130,Jamaica,Queens,Boro Zone
121,Hillcrest/Pomonok,Queens,Boro Zone
17,Bedford,Brooklyn,Boro Zone
89,Flatbush/Ditmas Park,Brooklyn,Boro Zone
128,Inwood Hill Park,Manhattan,Boro Zone
211,SoHo,Manhattan,Yellow Zone
203,Rosedale,Queens,Boro Zone
176,Oakwood,Staten Island,Boro Zone
221,Stapleton,Staten Island,Boro Zone
